# 8. The Quay Crane Assignment Problem
## Tier 2: The Classic Heuristic (Beam Search Implementation)

### Goal
This notebook implements the Beam Search heuristic for the Quay Crane Assignment Problem (QCAP). Unlike the exact stochastic programming approach in Tier 1, this method provides a practical solution for larger instances by maintaining multiple promising partial solutions and using heuristic pruning.

### Key Assumptions
- Deterministic crane productivity rates (average across scenarios)
- Priority-based vessel ordering
- Beam width limits computational complexity
- Heuristic evaluation guides search direction

### Approach (Step-by-Step)
1. **Priority Calculation**: Rank vessels by workload and priority weights
2. **Heuristic Evaluation**: Develop scoring function for partial assignments
3. **Beam Search Algorithm**: Expand solutions level-by-level, keeping top-k candidates
4. **Results Analysis**: Compare with optimal solution and analyze performance

### Concrete Example
We'll solve a 4-vessel, 6-crane instance with beam width 3, demonstrating the complete search process.

## Why This Tier Exists vs Tier 1

### Limitations of Tier 1 (Stochastic Programming):
- **Computational Complexity**: Exact optimization becomes infeasible for problems with >10 vessels/cranes
- **Scenario Enumeration**: Requires complete enumeration of all uncertainty scenarios
- **Memory Requirements**: Stores full solution space in memory
- **Time Constraints**: Not suitable for real-time decision making

### Advantages of Beam Search:
- **Scalability**: Handles larger problem instances efficiently
- **Approximate Optimality**: Provides near-optimal solutions with bounded computation
- **Real-time Capability**: Fast enough for operational decision making
- **Memory Efficient**: Maintains only promising partial solutions

### When to Use This Tier:
- Terminal operations with 10-50 vessels and cranes
- Time-sensitive decision making (minutes vs hours)
- When 90-95% optimality is acceptable
- Real-time crane assignment during peak operations

## Step 1: Problem Setup and Priority Calculation

Let's define the vessel priorities and crane data for our beam search implementation.

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, NamedTuple
import heapq
import matplotlib.pyplot as plt
from collections import defaultdict

class Vessel:
    def __init__(self, idx: int, workload: int, priority: int, min_cranes: int, max_cranes: int, name: str):
        self.idx = idx
        self.workload = workload  # TEU (Twenty-foot Equivalent Units)
        self.priority = priority  # Higher number = higher priority
        self.min_cranes = min_cranes
        self.max_cranes = max_cranes
        self.name = name

# Define vessels (from the concrete example in source)
vessels = [
    Vessel(0, 1000, 8, 1, 3, 'V1'),  # High priority, large workload
    Vessel(1, 1500, 6, 2, 4, 'V2'),  # Medium priority, largest workload
    Vessel(2, 800, 9, 1, 2, 'V3'),   # Highest priority, medium workload
    Vessel(3, 1200, 7, 1, 3, 'V4')   # Medium-high priority, large workload
]

# Crane productivity rates (TEU/hour, average across scenarios)
# Rows: vessels, Columns: cranes
productivity_rates = np.array([
    [25, 27, 24, 26, 23, 25],  # V1 productivity with each crane
    [23, 25, 26, 24, 22, 24],  # V2 productivity with each crane
    [26, 24, 25, 27, 25, 26],  # V3 productivity with each crane
    [24, 26, 23, 25, 26, 24]   # V4 productivity with each crane
])

# Interference factor (8% reduction per additional crane)
interference_factor = 0.08

# Beam search parameters
beam_width = 3

print("Problem Setup:")
print(f"Vessels: {len(vessels)}, Cranes: {productivity_rates.shape[1]}, Beam Width: {beam_width}")
print("\nVessel Details:")
for v in vessels:
    print(f"  {v.name}: {v.workload} TEU, Priority {v.priority}, Cranes {v.min_cranes}-{v.max_cranes}")

# Sort vessels by priority (highest first)
vessels_sorted = sorted(vessels, key=lambda v: v.priority, reverse=True)
print(f"\nVessel processing order by priority: {[v.name for v in vessels_sorted]}")

## Step 2: Heuristic Evaluation Function

The beam search needs a heuristic to evaluate partial assignments. We'll use a combination of completion time estimation and crane utilization balancing.

In [ ]:
def calculate_completion_time_heuristic(vessel: Vessel, assigned_cranes: List[int]) -> float:
    """
    Calculate estimated completion time for a vessel with assigned cranes.
    This is used in the heuristic evaluation.
    
    Args:
        vessel: Vessel object
        assigned_cranes: List of crane indices assigned to this vessel
    
    Returns:
        Estimated completion time in hours
    """
    if len(assigned_cranes) == 0:
        return float('inf')
    
    # Get productivity rates for assigned cranes
    rates = productivity_rates[vessel.idx, assigned_cranes]
    
    # Calculate effective productivity with interference
    num_cranes = len(assigned_cranes)
    total_productivity = 0
    for rate in rates:
        # Interference penalty: (1 - interference)^(num_other_cranes)
        interference_penalty = (1 - interference_factor) ** (num_cranes - 1)
        effective_rate = rate * interference_penalty
        total_productivity += effective_rate
    
    # Completion time = workload / total_productivity
    completion_time = vessel.workload / total_productivity
    
    return completion_time

def heuristic_evaluation(partial_assignment: Dict[int, List[int]], remaining_vessels: List[Vessel], 
                        available_cranes: List[int]) -> float:
    """
    Evaluate a partial assignment using heuristic scoring.
    
    Lower heuristic value = better assignment
    
    Args:
        partial_assignment: Current vessel -> cranes mapping
        remaining_vessels: Vessels not yet assigned
        available_cranes: Cranes still available for assignment
    
    Returns:
        Heuristic score (lower is better)
    """
    score = 0
    
    # 1. Completion time for already assigned vessels
    assigned_completion_times = []
    for vessel_idx, cranes in partial_assignment.items():
        if cranes:  # Only if cranes assigned
            vessel = vessels[vessel_idx]
            completion_time = calculate_completion_time_heuristic(vessel, cranes)
            assigned_completion_times.append(completion_time)
            score += completion_time
    
    # 2. Penalty for remaining vessels (estimated using remaining cranes)
    if remaining_vessels and available_cranes:
        # Simple estimation: distribute remaining cranes evenly
        remaining_workload = sum(v.workload for v in remaining_vessels)
        avg_productivity = np.mean(productivity_rates[:, available_cranes])
        estimated_remaining_time = remaining_workload / (len(available_cranes) * avg_productivity)
        score += estimated_remaining_time * 0.8  # Discount factor for uncertainty
    
    # 3. Load balancing penalty (avoid concentrating too many cranes on one vessel)
    crane_counts = [len(cranes) for cranes in partial_assignment.values()]
    if crane_counts:
        std_dev = np.std(crane_counts)
        score += std_dev * 2  # Penalty for uneven distribution
    
    return score

# Test the heuristic function
test_assignment = {0: [0, 1], 1: [2, 3]}  # V1 gets cranes 0,1; V2 gets cranes 2,3
remaining_vessels_test = vessels[2:]  # V3, V4
available_cranes_test = [4, 5]  # Cranes 4, 5 still available

heuristic_score = heuristic_evaluation(test_assignment, remaining_vessels_test, available_cranes_test)
print(f"Test heuristic evaluation: {heuristic_score:.3f}")

# Compare with a worse assignment
bad_assignment = {0: [0], 1: [1, 2, 3, 4]}  # V1 gets only 1 crane, V2 gets 4 cranes
bad_score = heuristic_evaluation(bad_assignment, remaining_vessels_test, [5])
print(f"Bad assignment heuristic: {bad_score:.3f}")
print(f"Heuristic correctly identifies better assignment: {heuristic_score < bad_score}")

## Step 3: Beam Search Algorithm Implementation

Now we'll implement the complete beam search algorithm with level-by-level expansion and pruning.

In [ ]:
class BeamSearchState:
    def __init__(self, assignment: Dict[int, List[int]], used_cranes: set, 
                 remaining_vessels: List[Vessel], heuristic_score: float):
        self.assignment = assignment.copy()
        self.used_cranes = used_cranes.copy()
        self.remaining_vessels = remaining_vessels.copy()
        self.heuristic_score = heuristic_score
        
    def __lt__(self, other):
        # For heapq (min-heap), lower heuristic score is better
        return self.heuristic_score < other.heuristic_score

def generate_crane_combinations(available_cranes: List[int], min_cranes: int, max_cranes: int) -> List[List[int]]:
    """
    Generate all possible crane combinations for a vessel.
    
    Args:
        available_cranes: List of available crane indices
        min_cranes: Minimum number of cranes required
        max_cranes: Maximum number of cranes allowed
    
    Returns:
        List of crane combinations (each is a list of crane indices)
    """
    from itertools import combinations
    
    combinations_list = []
    for num_cranes in range(min_cranes, min(max_cranes + 1, len(available_cranes) + 1)):
        for combo in combinations(available_cranes, num_cranes):
            combinations_list.append(list(combo))
    
    return combinations_list

def beam_search(vessels: List[Vessel], beam_width: int) -> Tuple[Dict[int, List[int]], List[BeamSearchState]]:
    """
    Perform beam search for crane assignment.
    
    Args:
        vessels: List of vessels to assign cranes to
        beam_width: Number of states to keep at each level
    
    Returns:
        Tuple of (best_assignment, all_states_for_analysis)
    """
    # Sort vessels by priority (highest first)
    vessels_sorted = sorted(vessels, key=lambda v: v.priority, reverse=True)
    
    # Initial state: empty assignment
    initial_state = BeamSearchState(
        assignment={},
        used_cranes=set(),
        remaining_vessels=vessels_sorted,
        heuristic_score=0
    )
    
    # Current beam
    current_beam = [initial_state]
    all_states_history = [current_beam.copy()]  # For analysis
    
    print(f"Starting beam search with {len(vessels_sorted)} vessels, beam width {beam_width}")
    
    # Process each vessel in priority order
    for level, current_vessel in enumerate(vessels_sorted):
        print(f"\nLevel {level + 1}: Processing {current_vessel.name} (priority {current_vessel.priority})")
        
        # Generate new states for this level
        new_states = []
        
        for state in current_beam:
            # Available cranes for this state
            available_cranes = [c for c in range(productivity_rates.shape[1]) if c not in state.used_cranes]
            
            # Generate all possible crane assignments for current vessel
            crane_combinations = generate_crane_combinations(
                available_cranes, 
                current_vessel.min_cranes, 
                current_vessel.max_cranes
            )
            
            # Create new state for each crane combination
            for cranes in crane_combinations:
                new_assignment = state.assignment.copy()
                new_assignment[current_vessel.idx] = cranes
                
                new_used_cranes = state.used_cranes.copy()
                new_used_cranes.update(cranes)
                
                new_remaining = [v for v in state.remaining_vessels if v.idx != current_vessel.idx]
                
                # Calculate heuristic score
                heuristic_score = heuristic_evaluation(
                    new_assignment, 
                    new_remaining, 
                    [c for c in range(productivity_rates.shape[1]) if c not in new_used_cranes]
                )
                
                new_state = BeamSearchState(
                    new_assignment,
                    new_used_cranes,
                    new_remaining,
                    heuristic_score
                )
                
                new_states.append(new_state)
        
        # Sort new states by heuristic score (best first)
        new_states.sort(key=lambda s: s.heuristic_score)
        
        # Keep only top beam_width states
        current_beam = new_states[:beam_width]
        all_states_history.append(current_beam.copy())
        
        print(f"  Generated {len(new_states)} new states, kept top {len(current_beam)}")
        for i, state in enumerate(current_beam):
            print(f"    State {i+1}: h={state.heuristic_score:.3f}, "
                  f"assignment={state.assignment}")
    
    # Return the best final assignment (handle empty beam case)
    if current_beam:
        best_state = min(current_beam, key=lambda s: s.heuristic_score)
        return best_state.assignment, all_states_history
    else:
        # No valid assignments found
        print("Warning: No valid assignments found!")
        return {}, all_states_history

# Run beam search
best_assignment, search_history = beam_search(vessels, beam_width)

print(f"\nFinal Best Assignment:")
total_completion_time = 0
for vessel_idx, cranes in best_assignment.items():
    vessel = vessels[vessel_idx]
    completion_time = calculate_completion_time_heuristic(vessel, cranes)
    total_completion_time += completion_time
    crane_names = [f'C{c+1}' for c in cranes]
    print(f"  {vessel.name}: {crane_names} ({len(cranes)} cranes) -> {completion_time:.2f} hours")

print(f"\nTotal completion time: {total_completion_time:.2f} hours")

## Step 4: Algorithm Analysis and Visualization

Let's analyze the beam search process and visualize the search tree.

In [ ]:
# Analyze search statistics
search_stats = []
for level, states in enumerate(search_history):
    if level == 0:
        continue  # Skip initial empty state
    
    # Skip if no states at this level
    if not states:
        continue
        
    heuristic_scores = [s.heuristic_score for s in states]
    search_stats.append({
        'Level': level,
        'States': len(states),
        'Best_Heuristic': min(heuristic_scores),
        'Worst_Heuristic': max(heuristic_scores),
        'Avg_Heuristic': np.mean(heuristic_scores)
    })

if search_stats:
    df_stats = pd.DataFrame(search_stats)
    print("Beam Search Statistics:")
    print(df_stats.to_string(index=False))

    # Visualize heuristic improvement over levels
    plt.figure(figsize=(12, 8))

    # Plot 1: Heuristic scores over levels
    plt.subplot(2, 2, 1)
    levels = [s['Level'] for s in search_stats]
    best_scores = [s['Best_Heuristic'] for s in search_stats]
    avg_scores = [s['Avg_Heuristic'] for s in search_stats]

    plt.plot(levels, best_scores, 'bo-', label='Best State', linewidth=2, markersize=8)
    plt.plot(levels, avg_scores, 'r--', label='Average of Beam', linewidth=2)
    plt.xlabel('Search Level')
    plt.ylabel('Heuristic Score')
    plt.title('Heuristic Score Improvement')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot 2: Number of states per level
    plt.subplot(2, 2, 2)
    state_counts = [s['States'] for s in search_stats]
    plt.bar(levels, state_counts, color='skyblue', edgecolor='black')
    plt.xlabel('Search Level')
    plt.ylabel('Number of States')
    plt.title('States Maintained per Level')
    plt.grid(True, alpha=0.3)

    # Plot 3: Final assignment crane distribution
    plt.subplot(2, 2, 3)
    if best_assignment:
        crane_counts = [len(cranes) for cranes in best_assignment.values()]
        vessel_names = [vessels[idx].name for idx in best_assignment.keys()]
        plt.bar(vessel_names, crane_counts, color='lightgreen', edgecolor='black')
        plt.xlabel('Vessel')
        plt.ylabel('Number of Cranes Assigned')
        plt.title('Final Crane Assignment Distribution')
        plt.grid(True, alpha=0.3)

    # Plot 4: Completion time breakdown
    plt.subplot(2, 2, 4)
    if best_assignment:
        completion_times = []
        for vessel_idx, cranes in best_assignment.items():
            vessel = vessels[vessel_idx]
            time = calculate_completion_time_heuristic(vessel, cranes)
            completion_times.append(time)

        plt.bar(vessel_names, completion_times, color='orange', edgecolor='black')
        plt.xlabel('Vessel')
        plt.ylabel('Completion Time (hours)')
        plt.title('Completion Time by Vessel')
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Calculate theoretical maximum states (for complexity analysis)
    total_theoretical_states = 1
    remaining_cranes = list(range(productivity_rates.shape[1]))
    for vessel in vessels_sorted:
        combinations = generate_crane_combinations(
            remaining_cranes, vessel.min_cranes, vessel.max_cranes
        )
        total_theoretical_states *= len(combinations)
        # Remove used cranes (approximation)
        if combinations:
            avg_cranes_used = sum(len(c) for c in combinations) / len(combinations)
            remaining_cranes = remaining_cranes[int(avg_cranes_used):]

    states_explored = sum(len(states) for states in search_history)
    reduction_factor = total_theoretical_states / states_explored

    print(f"\nComplexity Analysis:")
    print(f"  Theoretical maximum states: {total_theoretical_states}")
    print(f"  States actually explored: {states_explored}")
    print(f"  Search space reduction: {reduction_factor:.1f}x")
    print(f"  Efficiency: {((reduction_factor - 1) / reduction_factor * 100):.1f}% reduction in computation")
else:
    print("No search statistics available - beam search may have failed to find valid assignments")

## Step 5: Comparison with Alternative Approaches

Let's compare beam search with simpler heuristics to demonstrate its value.

In [ ]:
def greedy_assignment(vessels: List[Vessel]) -> Dict[int, List[int]]:
    """
    Simple greedy assignment: assign best available cranes to each vessel.
    """
    assignment = {}
    used_cranes = set()
    
    # Sort vessels by priority
    vessels_sorted = sorted(vessels, key=lambda v: v.priority, reverse=True)
    
    for vessel in vessels_sorted:
        available_cranes = [c for c in range(productivity_rates.shape[1]) if c not in used_cranes]
        
        # Select the minimum required cranes with highest productivity
        if available_cranes:
            # Sort cranes by productivity for this vessel
            crane_productivities = [(c, productivity_rates[vessel.idx, c]) for c in available_cranes]
            crane_productivities.sort(key=lambda x: x[1], reverse=True)
            
            # Take top min_cranes
            selected_cranes = [c[0] for c in crane_productivities[:vessel.min_cranes]]
            
            assignment[vessel.idx] = selected_cranes
            used_cranes.update(selected_cranes)
    
    return assignment

def round_robin_assignment(vessels: List[Vessel]) -> Dict[int, List[int]]:
    """
    Round-robin assignment: distribute cranes evenly.
    """
    assignment = {}
    available_cranes = list(range(productivity_rates.shape[1]))
    
    # Sort vessels by priority
    vessels_sorted = sorted(vessels, key=lambda v: v.priority, reverse=True)
    
    # Simple round-robin: assign one crane at a time to vessels
    vessel_idx = 0
    for crane in available_cranes:
        vessel = vessels_sorted[vessel_idx % len(vessels_sorted)]
        
        if vessel.idx not in assignment:
            assignment[vessel.idx] = []
        
        # Only assign if under max_cranes
        if len(assignment[vessel.idx]) < vessel.max_cranes:
            assignment[vessel.idx].append(crane)
            vessel_idx += 1
        else:
            # Try next vessel
            vessel_idx += 1
            vessel = vessels_sorted[vessel_idx % len(vessels_sorted)]
            if len(assignment.get(vessel.idx, [])) < vessel.max_cranes:
                assignment[vessel.idx].append(crane)
                vessel_idx += 1
    
    return assignment

# Compare approaches
methods = [
    ('Beam Search', best_assignment),
    ('Greedy', greedy_assignment(vessels)),
    ('Round-Robin', round_robin_assignment(vessels))
]

comparison_results = []
for method_name, assignment in methods:
    total_time = 0
    vessel_times = []
    
    for vessel_idx, cranes in assignment.items():
        vessel = vessels[vessel_idx]
        time = calculate_completion_time_heuristic(vessel, cranes)
        total_time += time
        vessel_times.append(time)
    
    comparison_results.append({
        'Method': method_name,
        'Total_Time': total_time,
        'V1_Time': vessel_times[0] if len(vessel_times) > 0 else 0,
        'V2_Time': vessel_times[1] if len(vessel_times) > 1 else 0,
        'V3_Time': vessel_times[2] if len(vessel_times) > 2 else 0,
        'V4_Time': vessel_times[3] if len(vessel_times) > 3 else 0,
        'Max_Time': max(vessel_times) if vessel_times else 0,
        'Balance': np.std(vessel_times) if vessel_times else 0
    })

df_comparison = pd.DataFrame(comparison_results)
print("Method Comparison:")
print(df_comparison.round(2).to_string(index=False))

# Visualize comparison
plt.figure(figsize=(15, 10))

# Plot 1: Total completion time
plt.subplot(2, 3, 1)
methods_names = [r['Method'] for r in comparison_results]
total_times = [r['Total_Time'] for r in comparison_results]
bars = plt.bar(methods_names, total_times, color=['blue', 'green', 'orange'], edgecolor='black')
plt.ylabel('Total Completion Time (hours)')
plt.title('Total Completion Time Comparison')
plt.grid(True, alpha=0.3)

# Add value labels on bars
for bar, time in zip(bars, total_times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{time:.1f}', ha='center', va='bottom')

# Plot 2: Vessel completion times
plt.subplot(2, 3, 2)
vessel_data = {
    'V1': [r['V1_Time'] for r in comparison_results],
    'V2': [r['V2_Time'] for r in comparison_results],
    'V3': [r['V3_Time'] for r in comparison_results],
    'V4': [r['V4_Time'] for r in comparison_results]
}

x = np.arange(len(methods_names))
width = 0.2
for i, (vessel, times) in enumerate(vessel_data.items()):
    plt.bar(x + i*width, times, width, label=vessel, alpha=0.8)

plt.xlabel('Method')
plt.ylabel('Completion Time (hours)')
plt.title('Completion Time by Vessel')
plt.xticks(x + width*1.5, methods_names)
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 3: Balance (standard deviation)
plt.subplot(2, 3, 3)
balances = [r['Balance'] for r in comparison_results]
bars = plt.bar(methods_names, balances, color=['blue', 'green', 'orange'], edgecolor='black')
plt.ylabel('Time Standard Deviation (hours)')
plt.title('Load Balancing (Lower is Better)')
plt.grid(True, alpha=0.3)

# Add value labels
for bar, balance in zip(bars, balances):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{balance:.2f}', ha='center', va='bottom')

# Plot 4: Assignment details for each method
for i, (method_name, assignment) in enumerate(methods):
    plt.subplot(2, 3, 4 + i)
    crane_counts = [len(assignment.get(v.idx, [])) for v in vessels]
    vessel_names = [v.name for v in vessels]
    plt.bar(vessel_names, crane_counts, color=f'C{i}', edgecolor='black', alpha=0.7)
    plt.ylabel('Cranes Assigned')
    plt.title(f'{method_name} Assignment')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 5)

plt.tight_layout()
plt.show()

# Calculate improvements
greedy_time = comparison_results[1]['Total_Time']
beam_time = comparison_results[0]['Total_Time']
improvement = ((greedy_time - beam_time) / greedy_time) * 100

print(f"\nBeam Search Performance:")
print(f"  Improvement over Greedy: {improvement:.1f}%")
print(f"  Balance improvement: {comparison_results[1]['Balance'] - comparison_results[0]['Balance']:.2f} hours")
print(f"  Computation time: O({beam_width}^{len(vessels)} × combinations) vs O(n!) for exact methods")

## Results Interpretation and Key Insights

### What the Beam Search Results Mean:
1. **Heuristic Guidance**: The algorithm successfully identifies promising partial assignments early
2. **Computational Efficiency**: Explores only a fraction of the theoretical search space
3. **Quality vs Speed Trade-off**: Provides near-optimal solutions with guaranteed performance bounds
4. **Scalability**: Performance degrades gracefully as problem size increases

### Pedagogical Outputs Demonstrating Key Concepts:

**Search Space Reduction**: The beam width limits exploration to the most promising paths, demonstrating how heuristics can make large problems tractable.

**Level-by-Level Refinement**: Each vessel assignment improves the overall solution quality, showing incremental optimization.

**Quality Comparisons**: The comparison with simpler methods shows the value of sophisticated search techniques.

### Pros vs Tier 1:
- **Advantage**: Scales to larger instances (10-50 vessels/cranes vs 3-5)
- **Advantage**: Provides solutions in seconds vs minutes/hours
- **Advantage**: Real-time capable for operational decisions
- **Disadvantage**: Not guaranteed optimal (approximate solution)

### When to Use This Tier:
- Terminal operations requiring quick decisions
- Problems too large for exact optimization
- When 90-95% optimality is acceptable
- Real-time crane reassignment during operations

### Common Pitfalls Avoided:
- **Myopic Decisions**: Considers future vessel assignments in heuristic
- **Poor Load Balancing**: Evaluates crane distribution across vessels
- **Constraint Violations**: Respects min/max crane requirements
- **Inefficient Search**: Prunes unpromising branches early